In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Hello World: Tvůj první kvantový Circuit

Sestav [Bellův stav](https://en.wikipedia.org/wiki/Bell_state) (dva provázané qubity) a spusť ho třemi způsoby:

1. **Ideální simulace** — dokonalé výsledky, účet není potřeba
2. **Simulace se šumem** — simuluje reálné zařízení, účet není potřeba
3. **Skutečný kvantový hardware** — vyžaduje bezplatný IBM Quantum účet (postup nastavení níže)

## Sestav Circuit

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

qc.draw(output="mpl")

## Možnost 1: Ideální simulace (bez účtu)
Používá `StatevectorSampler` — lokální simulátor s dokonalými výsledky bez šumu.

In [ ]:
from qiskit.primitives import StatevectorSampler

result = StatevectorSampler().run([qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
from qiskit.visualization import plot_histogram
plot_histogram(counts)

## Možnost 2: Simulace se šumem (bez účtu)
Používá `FakeManilaV2` — lokální simulátor, který napodobuje reálné IBM kvantové zařízení včetně jeho šumových charakteristik. Circuit musí být nejprve transpilován (přizpůsoben) na sadu Gate a konektivitu Qubitů daného zařízení.

In [ ]:
from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = FakeManilaV2()
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## Možnost 3: Skutečný kvantový hardware
Vyžaduje bezplatný IBM Quantum účet. Jak si ho vytvořit:

1. Zaregistruj se na [quantum.cloud.ibm.com/registration](https://quantum.cloud.ibm.com/registration) — pro prvních 30 dní není vyžadována platební karta
2. Přihlas se na [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) a vyber region **us-east** (povinné pro bezplatný Open Plan)
3. Na stránce [Instances](https://quantum.cloud.ibm.com/instances) vytvoř instanci (bezplatný Open Plan), pokud ještě žádnou nemáš
4. Na [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) (nebo na [cloud.ibm.com/iam/apikeys](https://cloud.ibm.com/iam/apikeys)) vytvoř API klíč
5. Zkopíruj svůj **CRN** (Cloud Resource Name) ze stránky [Instances](https://quantum.cloud.ibm.com/instances)

Poté, pokud jsi své přihlašovací údaje v této Binder relaci ještě neuložil, spusť níže uvedenou buňku. Nahraď `<your-api-key>` API klíčem z kroku 4 a `<your-crn>` hodnotou CRN z kroku 5.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="<your-api-key>",
    instance="<your-crn>",
    set_as_default=True,
    overwrite=True,
)

**Poznámka:** Úlohy na skutečném hardwaru mohou trvat určitou dobu v závislosti na čekacích dobách fronty. Pokud buňka stále běží, můžeš zkontrolovat stav své úlohy a zobrazit výsledky na [quantum.cloud.ibm.com/workloads](https://quantum.cloud.ibm.com/workloads?user=me).

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f"Running on {backend.name}")

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## Co dál?
- **[Tutoriály](https://doqumentation.org/tutorials)** — podrobné průvodce algoritmy, potlačením chyb, transpilací a dalšími tématy
- **[Průvodci](https://doqumentation.org/guides)** — referenční materiály ke spouštění Circuit, primitiv a Qiskit Runtime
- **[Kurzy](https://doqumentation.org/learning/courses/basics-of-quantum-information)** — strukturované vzdělávací cesty od základů kvantového světa po výpočty v utility měřítku
- **[Moduly](https://doqumentation.org/learning/modules/computer-science)** — hlubší koncepční moduly z informatiky a kvantové mechaniky